# 04. Statistical Analysis
**Project:** Logistics & Delivery Delays Analysis (Olist)
**Focus:** Hypothesis testing specifically targeted at the D1–D4 dashboard requirements: geographic influence, seller accountability, ETA accuracy, and customer satisfaction damage.


In [25]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

PATH_CLEANED = "../data/processed/olist_cleaned_data.csv"
df = pd.read_csv(PATH_CLEANED)
df['is_bad_review'] = df['review_score'] == 1
print(f"Dataset loaded. Shape: {df.shape}")


Dataset loaded. Shape: (109657, 44)


## 1. D4 Alignment: T-Test for Customer Impact
**Question:** Do late orders have statistically worse review scores than on-time orders?
**Dashboard:** D4 (Customer Experience Impact)


## 1.1 Why T-Test for On-Time vs Late Delays?

We have two groups:
- On-time orders: actual_delivery <= estimated_delivery (delay_days <= 0)
- Late orders: actual_delivery > estimated_delivery (delay_days > 0)

Question: Are these two groups statistically DIFFERENT in delay magnitude?
- Null hypothesis: Both groups have the same average delay
- If p-value < 0.05: Groups are significantly different (reject null)
- Business implication: On-time orders are NOT randomly similar to late orders

Why this matters:
- Confirms that "delay_days" metric is meaningful
- Validates that we can treat these as two distinct populations
- Justifies separate analysis of each group

In [26]:
late_reviews   = df[df['is_late'] == True]['review_score'].dropna()
ontime_reviews = df[df['is_late'] == False]['review_score'].dropna()

t_stat, t_pval = stats.ttest_ind(ontime_reviews, late_reviews, equal_var=False)

print(f"T-Test (Review Scores: On-Time vs Late)")
print(f"  On-Time avg  : {ontime_reviews.mean():.2f}")
print(f"  Late avg     : {late_reviews.mean():.2f}")
print(f"  T-Statistic  : {t_stat:.4f}")
print(f"  P-Value      : {t_pval:.4e}")
if t_pval < 0.05:
    print("  Conclusion   : REJECT H0. Late deliveries significantly degrade customer satisfaction.")
else:
    print("  Conclusion   : Fail to reject H0.")


T-Test (Review Scores: On-Time vs Late)
  On-Time avg  : 4.21
  Late avg     : 2.48
  T-Statistic  : 89.1575
  P-Value      : 0.0000e+00
  Conclusion   : REJECT H0. Late deliveries significantly degrade customer satisfaction.


## 1.2 T-Test Results Analysis

Interpreting the t-statistic and p-value:
- t-statistic: Magnitude of difference between groups (higher = bigger difference)
- p-value: Probability this difference is due to random chance
- If p-value < 0.001: EXTREMELY significant difference (almost 0% chance of randomness)
- Effect size: Mean difference between on-time and late groups

Business conclusion:
- Confirms our metrics are meaningful
- On-time and late orders are fundamentally different populations
- Can confidently use delay_days for downstream analysis

## 2.1 Why Chi-Square for Late Rate Distribution?

Chi-square tests: Does observed frequency match expected frequency?

Question: Is the late delivery rate significantly different than what we'd expect by chance?
- Null hypothesis: Late rate is consistent across all orders
- Alternative: Late rate varies (some have higher late %, some lower)

This reveals:
- Whether delays are sporadic (random) or systematic (pattern)
- If patterns exist, where to investigate

## 2. D4 Alignment: Chi-Square for Bad Reviews
**Question:** Is the likelihood of receiving a 1-star "bad review" statistically dependent on whether the delivery was late?
**Dashboard:** D4 (Customer Experience Impact)


## 2.2 Chi-Square Results Analysis

Chi-square statistic and p-value interpretation:
- If p-value < 0.05: Late rate distribution is NOT random
- Chi-square value: Measures how far observed differs from expected
- Higher chi-square: More pronounced pattern in the data

Business conclusion:
- If significant: Specific groups/routes have genuinely worse late rates
- If not significant: Late rates are evenly distributed (systemic problem)
- Guides investigation: Look for patterns or assume company-wide issue

## 3.1 Why ANOVA for State-Level Delays?

ANOVA (Analysis of Variance) compares MULTIPLE groups (more than 2):
- Groups: Different seller states (SP, RJ, MG, etc.)
- Question: Do different states have significantly different average delays?
- Null hypothesis: All states have same average delay
- If rejected: Some states genuinely perform worse

Why test this:
- Confirms geographic variation is real, not random
- Justifies regional intervention strategies
- Identifies which states need immediate attention

In [27]:
contingency_table = pd.crosstab(df['is_bad_review'], df['is_late'])

chi2_stat, chi2_pval, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi-Square Test (1-Star Review vs Delivery Lateness)")
print(f"  Chi-Square Statistic : {chi2_stat:.4f}")
print(f"  P-Value              : {chi2_pval:.4e}")
if chi2_pval < 0.05:
    print("  Conclusion           : REJECT H0. Receiving a 1-star review is statistically dependent on late delivery.")
else:
    print("  Conclusion           : Fail to reject H0.")


Chi-Square Test (1-Star Review vs Delivery Lateness)
  Chi-Square Statistic : 10990.6832
  P-Value              : 0.0000e+00
  Conclusion           : REJECT H0. Receiving a 1-star review is statistically dependent on late delivery.


## 3.2 ANOVA Results Analysis

F-statistic and p-value interpretation:
- F-statistic: Ratio of between-group variation to within-group variation
- Higher F: More differences between states
- p-value < 0.05: At least ONE state is significantly different

If significant:
- Post-hoc test (Tukey): Compare specific state pairs
- Identify worst and best performing states
- Develop state-specific interventions

Decision:
- Focus resources on worst-performing states
- Use best-performing states as operational benchmarks

## 4.1 Why Correlation for Distance vs Delay?

Pearson correlation tests: Do two variables move together?
- Variables: Geographic distance (km) and delay_days
- Question: Does longer distance predict longer delays?
- Null hypothesis: No correlation (r = 0)
- If rejected: Distance and delay are linked

Business logic:
- Farther shipments require more time
- If no correlation: Carrier/logistics problems, not distance
- If strong correlation: Geographic factors drive delays
- If weak correlation: Operational factors matter more

## 3. D2 Alignment: ANOVA for Geographic Variance
**Question:** Do different seller states have statistically significant differences in delivery delay?
**Dashboard:** D2 (Geographic Risk Monitor)


## 4.2 Correlation Analysis Results

Pearson r and p-value interpretation:
- r ranges from -1 (perfect negative) to +1 (perfect positive)
- |r| interpretation:
  - 0.7-1.0: Strong correlation
  - 0.4-0.7: Moderate correlation
  - 0.1-0.4: Weak correlation
  - 0.0-0.1: Very weak/no correlation
- p-value < 0.05: Correlation is statistically significant

Business implications:
- Strong positive: Distance matters, focus on route optimization
- Weak: Distance matters less, focus on operational improvements
- Negative: Counterintuitive, worth investigating

## 5.1 Why ETA Accuracy Validation?

Estimated Delivery Dates (ETA) are critical:
- Set customer expectations
- Drive customer satisfaction
- Indicate operational control

Analysis: How accurate are our ETAs?
- If mostly early arrivals: We're over-estimating (conservative)
- If mostly late arrivals: We're under-estimating (risky)
- If balanced: ETAs are well-calibrated

Question: Is our estimation strategy (early/balanced/late) consistent?

In [28]:
top_5_states = df['seller_state'].value_counts().head(5).index
anova_df = df[df['seller_state'].isin(top_5_states)][['seller_state', 'delivery_delay']].dropna()
state_groups = [group['delivery_delay'].values for _, group in anova_df.groupby('seller_state')]

f_stat, anova_pval = stats.f_oneway(*state_groups)

print(f"ANOVA Test (Delivery Delay Across Top 5 Seller States)")
print(f"  F-Statistic : {f_stat:.4f}")
print(f"  P-Value     : {anova_pval:.4e}")
if anova_pval < 0.05:
    print("  Conclusion  : REJECT H0. Seller state fundamentally influences delivery speed.")
else:
    print("  Conclusion  : Fail to reject H0.")


ANOVA Test (Delivery Delay Across Top 5 Seller States)
  F-Statistic : 397.0750
  P-Value     : 0.0000e+00
  Conclusion  : REJECT H0. Seller state fundamentally influences delivery speed.


## 5.2 ETA Accuracy Analysis Results

Delay distribution statistics:
- Mean (average) delay_days:
  - Negative: On average, we promise later than needed (safe)
  - Positive: On average, we promise earlier than we deliver (risky)
- Std Dev: How much variation (±consistency)
- Median: 50th percentile (robust to outliers)

Percentile analysis (25th, 50th, 75th):
- Shows spread of delay_days across customer base
- 10th percentile: Best-case scenario
- 90th percentile: Worst-case scenario
- Range = Operational variability

Interpretation:
- If range is narrow: Consistent performance
- If range is wide: High variability (needs investigation)
- If mean is very negative: Opportunity to reduce buffer (set tighter ETAs)